# Agentic Sidecar Universal Demo

This notebook explains what `agentic-sidecar` is used for, why it is required, and how the same pattern can supervise many kinds of autonomous-agent workflows.

`agentic-sidecar` is a pre-action supervision layer: it evaluates a proposed agent action before that action reaches a real tool, API, database, or external system.

The core question is:

> Should this agent do this action right now, given the user's original intent, policy, and risk tolerance?

This demo uses the implemented core API directly, so it runs without LangGraph, an LLM provider, or an API key.

## Why Teams Add A Sidecar

Autonomous agents often make many decisions between a user request and a real-world effect. Basic permission checks usually answer only: "Can this tool be called?"

The sidecar answers a more useful governance question:

- Is the action permitted by policy?
- Is the action too risky for the configured threshold?
- Does the action still match the user's original constraints?
- Should the action be allowed, warned about, or blocked before execution?

This matters because an action can be technically available but contextually wrong. An agent may have access to `deploy_service`, `send_email`, `issue_refund`, or `query_database`, but the user may have authorized only a narrow version of that work.

In [12]:
from dataclasses import dataclass
from collections.abc import Callable

from agentic_sidecar import Sidecar
from agentic_sidecar.core.context import DecisionContext
from agentic_sidecar.gate.policy import PolicyAdvisor, PolicyRule
from agentic_sidecar.gate.risk import RiskEvaluator, RiskRule
from agentic_sidecar.intent.alignment import ConstraintBinding, IntentGuardian
from agentic_sidecar.intent.envelope import IntentEnvelope, Requester

## The Universal Adoption Pattern

The same four-step pattern works across domains:

1. Capture the user's task as an `IntentEnvelope`.
2. Express important limits as structured `constraints`.
3. Bind each constraint to the relevant tool argument using `ConstraintBinding`.
4. Evaluate every proposed tool call through `Sidecar.evaluate()` before execution.

In a real framework integration, an adapter wraps the tools. In this notebook, we call `evaluate()` directly to make the behavior easy to see.

In [13]:
@dataclass(frozen=True)
class DemoScenario:
    name: str
    user_request: str
    envelope: IntentEnvelope
    bindings: list[ConstraintBinding]
    actions: list[DecisionContext]


def build_sidecar(scenario: DemoScenario) -> Sidecar:
    policy = PolicyAdvisor(
        [
            PolicyRule(
                tool="delete_*",
                effect="deny",
                reason="Destructive operations require separate human approval.",
            ),
            PolicyRule(
                tool="send_external_email",
                effect="deny",
                reason="External email is disabled for this demo tenant.",
            ),
        ]
    )

    risk = RiskEvaluator(
        [
            RiskRule(tool="delete_*", risk="HIGH", reason="Deletion is a high-risk operation."),
            RiskRule(tool="deploy_*", risk="MEDIUM", reason="Deployments can change runtime behavior."),
            RiskRule(tool="issue_refund", arg_name="amount", arg_op="gt", arg_value=500, risk="MEDIUM"),
            RiskRule(tool="query_database", risk="MEDIUM", reason="Database queries may expose sensitive data."),
        ]
    )

    guardian = IntentGuardian(scenario.envelope, scenario.bindings)
    return Sidecar(
        on_sidecar_failure="fail_closed",
        roles=["policy", "risk", "intent_guardian"],
        policy=policy,
        risk=risk,
        intent=guardian,
        mode="govern",
    )


def run_scenario(scenario: DemoScenario) -> list[dict[str, str]]:
    sidecar = build_sidecar(scenario)
    rows = []
    print(f"SCENARIO: {scenario.name}")
    print(f"USER REQUEST: {scenario.user_request}")
    print(f"INTENT CONSTRAINTS: {scenario.envelope.constraints}\n")

    for action in scenario.actions:
        decision = sidecar.evaluate(action)
        rows.append(
            {
                "scenario": scenario.name,
                "tool": action.tool_name,
                "args": str(action.tool_args),
                "decision": decision.status,
                "risk": decision.risk or "-",
                "reason": decision.reason,
            }
        )
        print(f"{decision.status:>5} | risk={decision.risk or '-':<6} | {action.tool_name} {action.tool_args}")
        print(f"      {decision.reason}\n")
    return rows


def render_decision_table(rows: list[dict[str, str]]) -> None:
    print("| Scenario | Tool | Decision | Risk | Reason |")
    print("|---|---|---:|---:|---|")
    for row in rows:
        reason = row["reason"].replace("|", "-")
        print(f"| {row['scenario']} | `{row['tool']}` | {row['decision']} | {row['risk']} | {reason} |")


def simulate_tool_execution(
    sidecar: Sidecar,
    action: DecisionContext,
    tool: Callable[..., str],
) -> str:
    decision = sidecar.evaluate(action)
    if sidecar.mode == "govern" and decision.status == "BLOCK":
        return f"Tool did not run. Sidecar blocked it: {decision.reason}"
    result = tool(**action.tool_args)
    return f"Tool ran. Decision was {decision.status}. Result: {result}"

## Scenario 1: Customer Support

The user authorizes a refund, but only up to $500. The agent can still read the order and issue a smaller refund, but an over-limit refund is blocked because it violates the user's intent.

In [14]:
support_scenario = DemoScenario(
    name="Customer support refund",
    user_request="Refund order A100, but do not refund more than $500.",
    envelope=IntentEnvelope(
        goal="refund_customer",
        requested_by=Requester(type="human", id="support_lead"),
        constraints={"maximum_refund": 500},
    ),
    bindings=[
        ConstraintBinding(
            constraint="maximum_refund",
            tool="issue_refund",
            arg_name="amount",
            op="lte",
        )
    ],
    actions=[
        DecisionContext(tool_name="read_order", tool_args={"order_id": "A100"}),
        DecisionContext(tool_name="issue_refund", tool_args={"order_id": "A100", "amount": 120}),
        DecisionContext(tool_name="issue_refund", tool_args={"order_id": "A100", "amount": 850}),
    ],
)

all_rows = []
all_rows.extend(run_scenario(support_scenario))

[GOVERN] BLOCK tool=issue_refund args={'order_id': 'A100', 'amount': 850} risk=MEDIUM reason=Intent Guardian: 'amount'=850 on 'issue_refund' violates constraint 'maximum_refund' (must be lte 500)


SCENARIO: Customer support refund
USER REQUEST: Refund order A100, but do not refund more than $500.
INTENT CONSTRAINTS: {'maximum_refund': 500}

ALLOW | risk=LOW    | read_order {'order_id': 'A100'}
      Policy Advisor: No policy rule matched tool 'read_order'; default effect is 'allow'; Risk Evaluator: No risk rule matched tool 'read_order'; default risk is 'LOW'; Intent Guardian: No intent constraints violated for 'read_order'

ALLOW | risk=LOW    | issue_refund {'order_id': 'A100', 'amount': 120}
      Policy Advisor: No policy rule matched tool 'issue_refund'; default effect is 'allow'; Risk Evaluator: No risk rule matched tool 'issue_refund'; default risk is 'LOW'; Intent Guardian: No intent constraints violated for 'issue_refund'

BLOCK | risk=MEDIUM | issue_refund {'order_id': 'A100', 'amount': 850}
      Intent Guardian: 'amount'=850 on 'issue_refund' violates constraint 'maximum_refund' (must be lte 500)



## Scenario 2: Production Operations

The user asks the agent to investigate production, but not modify production. The sidecar lets read-only inspection proceed and blocks a production deployment because the action no longer matches the authorized environment.

In [15]:
ops_scenario = DemoScenario(
    name="Production investigation",
    user_request="Investigate the production issue. Do not modify production.",
    envelope=IntentEnvelope(
        goal="investigate_production_issue",
        requested_by=Requester(type="human", id="incident_commander"),
        constraints={"allowed_environment": "staging"},
    ),
    bindings=[
        ConstraintBinding(
            constraint="allowed_environment",
            tool="deploy_service",
            arg_name="environment",
            op="eq",
        )
    ],
    actions=[
        DecisionContext(tool_name="read_logs", tool_args={"environment": "production"}),
        DecisionContext(tool_name="deploy_service", tool_args={"service": "api", "environment": "staging"}),
        DecisionContext(tool_name="deploy_service", tool_args={"service": "api", "environment": "production"}),
        DecisionContext(tool_name="delete_database", tool_args={"environment": "production"}),
    ],
)

all_rows.extend(run_scenario(ops_scenario))

[GOVERN] BLOCK tool=deploy_service args={'service': 'api', 'environment': 'production'} risk=MEDIUM reason=Intent Guardian: 'environment'='production' on 'deploy_service' violates constraint 'allowed_environment' (must be eq 'staging')
[GOVERN] BLOCK tool=delete_database args={'environment': 'production'} risk=None reason=Policy Advisor: Destructive operations require separate human approval.


SCENARIO: Production investigation
USER REQUEST: Investigate the production issue. Do not modify production.
INTENT CONSTRAINTS: {'allowed_environment': 'staging'}

ALLOW | risk=LOW    | read_logs {'environment': 'production'}
      Policy Advisor: No policy rule matched tool 'read_logs'; default effect is 'allow'; Risk Evaluator: No risk rule matched tool 'read_logs'; default risk is 'LOW'; Intent Guardian: No intent constraints violated for 'read_logs'

ALLOW | risk=MEDIUM | deploy_service {'service': 'api', 'environment': 'staging'}
      Policy Advisor: No policy rule matched tool 'deploy_service'; default effect is 'allow'; Risk Evaluator: Deployments can change runtime behavior.; Intent Guardian: No intent constraints violated for 'deploy_service'

BLOCK | risk=MEDIUM | deploy_service {'service': 'api', 'environment': 'production'}
      Intent Guardian: 'environment'='production' on 'deploy_service' violates constraint 'allowed_environment' (must be eq 'staging')

BLOCK | risk=-

## Scenario 3: Research And Data Access

The user asks for research using approved public sources only. The sidecar allows an approved source and blocks a private database query because it violates the allowed-source constraint.

In [16]:
research_scenario = DemoScenario(
    name="Research with approved sources",
    user_request="Research the topic using public documentation only.",
    envelope=IntentEnvelope(
        goal="research_public_docs",
        requested_by=Requester(type="human", id="researcher"),
        constraints={"allowed_source": "public_docs"},
    ),
    bindings=[
        ConstraintBinding(
            constraint="allowed_source",
            tool="fetch_source",
            arg_name="source_type",
            op="eq",
        ),
        ConstraintBinding(
            constraint="allowed_source",
            tool="query_database",
            arg_name="source_type",
            op="eq",
        ),
    ],
    actions=[
        DecisionContext(tool_name="fetch_source", tool_args={"source_type": "public_docs", "domain": "docs.example.com"}),
        DecisionContext(tool_name="query_database", tool_args={"source_type": "private_customer_data", "table": "customers"}),
        DecisionContext(tool_name="send_external_email", tool_args={"recipient": "vendor@example.com"}),
    ],
)

all_rows.extend(run_scenario(research_scenario))

[GOVERN] BLOCK tool=query_database args={'source_type': 'private_customer_data', 'table': 'customers'} risk=MEDIUM reason=Intent Guardian: 'source_type'='private_customer_data' on 'query_database' violates constraint 'allowed_source' (must be eq 'public_docs')
[GOVERN] BLOCK tool=send_external_email args={'recipient': 'vendor@example.com'} risk=None reason=Policy Advisor: External email is disabled for this demo tenant.


SCENARIO: Research with approved sources
USER REQUEST: Research the topic using public documentation only.
INTENT CONSTRAINTS: {'allowed_source': 'public_docs'}

ALLOW | risk=LOW    | fetch_source {'source_type': 'public_docs', 'domain': 'docs.example.com'}
      Policy Advisor: No policy rule matched tool 'fetch_source'; default effect is 'allow'; Risk Evaluator: No risk rule matched tool 'fetch_source'; default risk is 'LOW'; Intent Guardian: No intent constraints violated for 'fetch_source'

BLOCK | risk=MEDIUM | query_database {'source_type': 'private_customer_data', 'table': 'customers'}
      Intent Guardian: 'source_type'='private_customer_data' on 'query_database' violates constraint 'allowed_source' (must be eq 'public_docs')

BLOCK | risk=-      | send_external_email {'recipient': 'vendor@example.com'}
      Policy Advisor: External email is disabled for this demo tenant.



## Visual Decision Table

For demos and reviews, a compact table is often easier to scan than raw logs.

In [17]:
render_decision_table(all_rows)

| Scenario | Tool | Decision | Risk | Reason |
|---|---|---:|---:|---|
| Customer support refund | `read_order` | ALLOW | LOW | Policy Advisor: No policy rule matched tool 'read_order'; default effect is 'allow'; Risk Evaluator: No risk rule matched tool 'read_order'; default risk is 'LOW'; Intent Guardian: No intent constraints violated for 'read_order' |
| Customer support refund | `issue_refund` | ALLOW | LOW | Policy Advisor: No policy rule matched tool 'issue_refund'; default effect is 'allow'; Risk Evaluator: No risk rule matched tool 'issue_refund'; default risk is 'LOW'; Intent Guardian: No intent constraints violated for 'issue_refund' |
| Customer support refund | `issue_refund` | BLOCK | MEDIUM | Intent Guardian: 'amount'=850 on 'issue_refund' violates constraint 'maximum_refund' (must be lte 500) |
| Production investigation | `read_logs` | ALLOW | LOW | Policy Advisor: No policy rule matched tool 'read_logs'; default effect is 'allow'; Risk Evaluator: No risk rule matched 

## Policy vs Risk vs Intent

These three checks answer different questions:

- Policy asks: is this action allowed by organizational rules?
- Risk asks: how dangerous is this action?
- Intent asks: does this action still match what the user authorized for this task?

Keeping them separate helps teams explain why an action was allowed or blocked.

In [18]:
comparison_scenario = DemoScenario(
    name="Policy, risk, and intent comparison",
    user_request="Investigate the app safely in staging only.",
    envelope=IntentEnvelope(
        goal="safe_app_investigation",
        requested_by=Requester(type="human", id="operator"),
        constraints={"allowed_environment": "staging"},
    ),
    bindings=[
        ConstraintBinding(
            constraint="allowed_environment",
            tool="deploy_service",
            arg_name="environment",
            op="eq",
        )
    ],
    actions=[
        DecisionContext(tool_name="delete_database", tool_args={"environment": "staging"}),
        DecisionContext(tool_name="deploy_service", tool_args={"environment": "staging"}),
        DecisionContext(tool_name="deploy_service", tool_args={"environment": "production"}),
    ],
)

run_scenario(comparison_scenario)

[GOVERN] BLOCK tool=delete_database args={'environment': 'staging'} risk=None reason=Policy Advisor: Destructive operations require separate human approval.
[GOVERN] BLOCK tool=deploy_service args={'environment': 'production'} risk=MEDIUM reason=Intent Guardian: 'environment'='production' on 'deploy_service' violates constraint 'allowed_environment' (must be eq 'staging')


SCENARIO: Policy, risk, and intent comparison
USER REQUEST: Investigate the app safely in staging only.
INTENT CONSTRAINTS: {'allowed_environment': 'staging'}

BLOCK | risk=-      | delete_database {'environment': 'staging'}
      Policy Advisor: Destructive operations require separate human approval.

ALLOW | risk=MEDIUM | deploy_service {'environment': 'staging'}
      Policy Advisor: No policy rule matched tool 'deploy_service'; default effect is 'allow'; Risk Evaluator: Deployments can change runtime behavior.; Intent Guardian: No intent constraints violated for 'deploy_service'

BLOCK | risk=MEDIUM | deploy_service {'environment': 'production'}
      Intent Guardian: 'environment'='production' on 'deploy_service' violates constraint 'allowed_environment' (must be eq 'staging')



[{'scenario': 'Policy, risk, and intent comparison',
  'tool': 'delete_database',
  'args': "{'environment': 'staging'}",
  'decision': 'BLOCK',
  'risk': '-',
  'reason': 'Policy Advisor: Destructive operations require separate human approval.'},
 {'scenario': 'Policy, risk, and intent comparison',
  'tool': 'deploy_service',
  'args': "{'environment': 'staging'}",
  'decision': 'ALLOW',
  'risk': 'MEDIUM',
  'reason': "Policy Advisor: No policy rule matched tool 'deploy_service'; default effect is 'allow'; Risk Evaluator: Deployments can change runtime behavior.; Intent Guardian: No intent constraints violated for 'deploy_service'"},
 {'scenario': 'Policy, risk, and intent comparison',
  'tool': 'deploy_service',
  'args': "{'environment': 'production'}",
  'decision': 'BLOCK',
  'risk': 'MEDIUM',
  'reason': "Intent Guardian: 'environment'='production' on 'deploy_service' violates constraint 'allowed_environment' (must be eq 'staging')"}]

## Observe Mode vs Govern Mode

Adoption usually starts in Observe mode. The sidecar records what it would decide, but your runtime does not have to stop the action yet.

Govern mode is stricter. A framework adapter can enforce `BLOCK` so the real tool does not run.

In [19]:
def dangerous_tool(environment: str) -> str:
    return f"changed {environment}"


action = DecisionContext(tool_name="deploy_service", tool_args={"environment": "production"})
observe_sidecar = build_sidecar(ops_scenario)
observe_sidecar.mode = "observe"

govern_sidecar = build_sidecar(ops_scenario)
govern_sidecar.mode = "govern"

print("Observe mode:")
print(simulate_tool_execution(observe_sidecar, action.model_copy(deep=True), dangerous_tool))

print("\nGovern mode:")
print(simulate_tool_execution(govern_sidecar, action.model_copy(deep=True), dangerous_tool))

[OBSERVE] BLOCK tool=deploy_service args={'environment': 'production'} risk=MEDIUM reason=Intent Guardian: 'environment'='production' on 'deploy_service' violates constraint 'allowed_environment' (must be eq 'staging')
[GOVERN] BLOCK tool=deploy_service args={'environment': 'production'} risk=MEDIUM reason=Intent Guardian: 'environment'='production' on 'deploy_service' violates constraint 'allowed_environment' (must be eq 'staging')


Observe mode:
Tool ran. Decision was BLOCK. Result: changed production

Govern mode:
Tool did not run. Sidecar blocked it: Intent Guardian: 'environment'='production' on 'deploy_service' violates constraint 'allowed_environment' (must be eq 'staging')


## YAML-Based Policy Example

Policies can also be loaded from mappings that mirror the YAML file shape. This is useful when teams want governance rules outside application code.

In [20]:
policy_config = {
    "default": "allow",
    "rules": [
        {
            "tool": "delete_*",
            "effect": "deny",
            "reason": "Delete operations require a separate approval workflow.",
        }
    ],
}

yaml_policy = PolicyAdvisor.from_mapping(policy_config)
result = yaml_policy.evaluate(DecisionContext(tool_name="delete_file", tool_args={"path": "report.csv"}))
print(result.effect)
print(result.reason)

deny
Delete operations require a separate approval workflow.


## Custom Use Case Cell

Edit the dictionary below to try your own domain. This keeps adoption concrete: define the user goal, one constraint, the tool to supervise, and the proposed tool arguments.

In [ ]:
custom_case = {
    "name": "Custom CRM workflow",
    "user_request": "Analyze customer feedback using survey exports only.",
    "goal": "analyze_customer_feedback",
    "constraint_name": "allowed_source",
    "constraint_value": "survey_exports",
    "tool": "query_database",
    "arg_name": "source_type",
    "allowed_action_args": {"source_type": "survey_exports", "table": "survey_responses"},
    "blocked_action_args": {"source_type": "crm_private_notes", "table": "customer_notes"},
}

custom_scenario = DemoScenario(
    name=custom_case["name"],
    user_request=custom_case["user_request"],
    envelope=IntentEnvelope(
        goal=custom_case["goal"],
        requested_by=Requester(type="human", id="custom_user"),
        constraints={custom_case["constraint_name"]: custom_case["constraint_value"]},
    ),
    bindings=[
        ConstraintBinding(
            constraint=custom_case["constraint_name"],
            tool=custom_case["tool"],
            arg_name=custom_case["arg_name"],
            op="eq",
        )
    ],
    actions=[
        DecisionContext(tool_name=custom_case["tool"], tool_args=custom_case["allowed_action_args"]),
        DecisionContext(tool_name=custom_case["tool"], tool_args=custom_case["blocked_action_args"]),
    ],
)

run_scenario(custom_scenario)

[GOVERN] BLOCK tool=query_database args={'source_type': 'crm_private_notes', 'table': 'customer_notes'} risk=MEDIUM reason=Intent Guardian: 'source_type'='crm_private_notes' on 'query_database' violates constraint 'allowed_source' (must be eq 'survey_exports')


SCENARIO: Custom CRM workflow
USER REQUEST: Analyze customer feedback using survey exports only.
INTENT CONSTRAINTS: {'allowed_source': 'survey_exports'}

ALLOW | risk=MEDIUM | query_database {'source_type': 'survey_exports', 'table': 'survey_responses'}
      Policy Advisor: No policy rule matched tool 'query_database'; default effect is 'allow'; Risk Evaluator: Database queries may expose sensitive data.; Intent Guardian: No intent constraints violated for 'query_database'

BLOCK | risk=MEDIUM | query_database {'source_type': 'crm_private_notes', 'table': 'customer_notes'}
      Intent Guardian: 'source_type'='crm_private_notes' on 'query_database' violates constraint 'allowed_source' (must be eq 'survey_exports')



[{'scenario': 'Custom CRM workflow',
  'tool': 'query_database',
  'args': "{'source_type': 'survey_exports', 'table': 'survey_responses'}",
  'decision': 'ALLOW',
  'risk': 'MEDIUM',
  'reason': "Policy Advisor: No policy rule matched tool 'query_database'; default effect is 'allow'; Risk Evaluator: Database queries may expose sensitive data.; Intent Guardian: No intent constraints violated for 'query_database'"},
 {'scenario': 'Custom CRM workflow',
  'tool': 'query_database',
  'args': "{'source_type': 'crm_private_notes', 'table': 'customer_notes'}",
  'decision': 'BLOCK',
  'risk': 'MEDIUM',
  'reason': "Intent Guardian: 'source_type'='crm_private_notes' on 'query_database' violates constraint 'allowed_source' (must be eq 'survey_exports')"}]

: 

## What This Shows

The sidecar is domain-independent. It does not know that refunds, deployments, or research are special. It evaluates structured proposed actions against three configurable layers:

- `PolicyAdvisor`: organization-level allow and deny rules.
- `RiskEvaluator`: deterministic risk classification rules.
- `IntentGuardian`: task-specific constraints derived from the user's request.

To adopt this in a new workflow, define the tools your agent can call, decide which tool arguments represent important user constraints, and bind those constraints to the sidecar.

## Adoption Template

For a new use case, fill in this shape:

```python
envelope = IntentEnvelope(
    goal="your_task_goal",
    requested_by=Requester(type="human", id="user_or_operator"),
    constraints={
        "constraint_name": "allowed_value",
    },
)

bindings = [
    ConstraintBinding(
        constraint="constraint_name",
        tool="tool_name_or_glob",
        arg_name="tool_argument_to_check",
        op="eq",
    )
]
```

The current implementation is intentionally deterministic. It responds according to the structured intent and bindings you provide. It does not yet infer every constraint from natural language automatically.

## What It Does Not Do Yet

The current implementation is intentionally deterministic and narrow:

- It does not automatically infer every constraint from free-form natural language.
- It does not yet implement Planner, Critic, Judge, Budget Guardian, human escalation, or replan outcomes.
- It does not yet provide every framework adapter. LangGraph is the first implemented adapter.
- It supervises tool-call decisions; it does not replace the agent runtime or execute the user's whole task.

That narrowness is useful for adoption because the decision behavior is predictable and explainable.

## Next: Real Agent Integration

This notebook calls `Sidecar.evaluate()` directly. To see the same concept attached to a real agent tool surface, look at the LangGraph examples in this repo:

- `examples/langgraph_refund_observe_mode.py`
- `examples/langgraph_intent_guardian_govern_mode.py`

Those examples show how `agentic_sidecar.adapters.langgraph.attach(sidecar, tools)` wraps tools so the sidecar evaluates calls before the tool function executes.

## Main Takeaway

`agentic-sidecar` is useful when an autonomous agent can take meaningful actions through tools. It adds a supervision checkpoint between "the agent decided" and "the external system changed."

Use it for workflows involving money, data changes, production systems, customer data, external communication, or delegated tasks where the original user intent can drift.